# Notebook 06 mathematical attention backend

Reusable definitions only. Inputs: existing semantic mixer Q/K/V, original grid indices, validity, fixed feature banks. Outputs: normalized attention and per-instance configuration. This module introduces differentiable key centering and a disjoint spatial tree without global-minus-local subtraction. It preserves the uncentered hybrid operator, while centering deliberately changes its finite random-feature approximation.

Run top-to-bottom in the `ahr-go` kernel; execution only defines functions (under 10 seconds). No training or checkpoints are created here. Experiment checkpoints and exact resume instructions belong to the Notebook 06 experiment companion. To reload definitions, restart the kernel and Run All. Only `ahr-export` cells are imported by other notebooks.

The tree uses sparse interaction matrices with at most 27 cells per receiver at radius 1. Local work is tiled in blocks of 32 receivers; internal tree/local tensors are recomputed during backward. A counted FP64 direct fallback handles underflow/overflow. No denominator clamp or approximation clipping is used. Tree memory scales with cell count × feature count × value width; improved performance is a hypothesis requiring measurement.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_namespace = {}
for _cell in json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _namespace)
_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Notebook06: centered random features and disjoint spatial tree accumulation."""
import copy
import importlib.util
import json
import math
import sys
import types
from pathlib import Path
import torch
from torch.utils.checkpoint import checkpoint
from fmm_jepa.semantic_attention import SemanticLocalMixer, _orthogonal_gaussian_features

backend = "semantic-math-v1"
_VARIANTS = ("stable", "centered", "tree", "centered_tree")
_TOPOLOGY = {}
_STATS = {"calls": 0, "valid_rows": 0, "fallback_calls": 0, "fallback_rows": 0}


def _stable_module():
    name = "fmm06_stable05_reference"
    if name not in sys.modules:
        root = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
        path = root / "experiments/05_scaling/stable_attention.ipynb"
        module = types.ModuleType(name); module.__file__ = str(path)
        sys.modules[name] = module
        try:
            for cell in json.loads(path.read_text())["cells"]:
                if cell["cell_type"] == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", []):
                    exec(compile("".join(cell["source"]), str(path) + "#ahr-export", "exec"), module.__dict__)
        except Exception:
            sys.modules.pop(name, None)
            raise
    return sys.modules[name]


def reset_stats():
    for key in _STATS:
        _STATS[key] = 0
    _stable_module().reset_stats()


def get_stats():
    return {**_STATS, "backend": backend, "fallback_fraction": _STATS["fallback_rows"] / max(1, _STATS["valid_rows"]), "stable_reference": _stable_module().get_stats()}


def center_keys(q, k, valid):
    """c=mean(q*d^-1/4)+mean(k*d^-1/4); return k-c*d^1/4.

    Means are differentiable. Padding is excluded before summing, and an empty
    sample uses zero center. The raw-space formula avoids redundant scale error.
    """
    active = valid[:, None, :, None]
    q_clean = torch.where(active, q, torch.zeros_like(q))
    k_clean = torch.where(active, k, torch.zeros_like(k))
    count = valid.sum(-1)[:, None, None, None].clamp_min(1).to(q.dtype)
    center_raw = (q_clean.sum(-2, keepdim=True) + k_clean.sum(-2, keepdim=True)) / count
    return torch.where(active, k_clean - center_raw, torch.zeros_like(k_clean))


def centered_mix_heads(self, q, k, v, indices, valid, grid):
    return _stable_module().stable_mix_heads(self, q, center_keys(q, k, valid), v, indices, valid, grid)


def _geometry(grid, leaf_size, radius, device, dtype):
    """Precompute sparse interaction lists; at radius1 each child has <=27 cells."""
    rows, cols = map(int, grid)
    leaf_rows, leaf_cols = math.ceil(rows / leaf_size), math.ceil(cols / leaf_size)
    side = 1 << (max(leaf_rows, leaf_cols) - 1).bit_length()
    key = (tuple(grid), leaf_size, radius, str(device), dtype)
    if key in _TOPOLOGY:
        return _TOPOLOGY[key]
    levels = []
    n = 2
    while n <= side:
        receivers, sources, parent = [], [], []
        maximum = 0
        for row in range(n):
            for col in range(n):
                target = row * n + col
                parent.append((row // 2) * (n // 2) + col // 2)
                start = len(sources)
                for pr in range(max(0, row // 2 - radius), min(n // 2, row // 2 + radius + 1)):
                    for pc in range(max(0, col // 2 - radius), min(n // 2, col // 2 + radius + 1)):
                        for dr in (0, 1):
                            for dc in (0, 1):
                                sr, sc = 2 * pr + dr, 2 * pc + dc
                                if max(abs(sr - row), abs(sc - col)) > radius:
                                    receivers.append(target); sources.append(sr * n + sc)
                maximum = max(maximum, len(sources) - start)
        coordinates = torch.tensor([receivers, sources], dtype=torch.long, device=device)
        interaction = torch.sparse_coo_tensor(coordinates, torch.ones(len(sources), dtype=dtype, device=device), (n*n, n*n)).coalesce()
        levels.append({"side": n, "interaction": interaction,
                       "parent": torch.tensor(parent, dtype=torch.long, device=device),
                       "max_interactions": maximum, "interaction_count": len(sources)})
        n *= 2
    result = {"side": side, "levels": levels, "leaf_slots": leaf_size ** 2,
              "cell_count": side ** 2, "leaf_rows": leaf_rows, "leaf_cols": leaf_cols}
    _TOPOLOGY[key] = result
    return result


def hierarchy_cost(grid, leaf_size=2, radius=1):
    geometry = _geometry(grid, leaf_size, radius, torch.device("cpu"), torch.float32)
    return {"leaf_cells": geometry["cell_count"], "leaf_slots": geometry["leaf_slots"],
            "total_cells": sum(4 ** level for level in range(len(geometry["levels"]) + 1)),
            "max_interaction_cells": max([0] + [level["max_interactions"] for level in geometry["levels"]]),
            "total_interactions": sum(level["interaction_count"] for level in geometry["levels"]),
            "local_sources_per_receiver_bound": (2 * radius + 1) ** 2 * leaf_size ** 2}


def _grid_slots(indices, valid, grid, leaf_size, side):
    row = indices.clamp_min(0).div(int(grid[1]), rounding_mode="floor")
    col = indices.clamp_min(0).remainder(int(grid[1]))
    cell = row.div(leaf_size, rounding_mode="floor") * side + col.div(leaf_size, rounding_mode="floor")
    within = row.remainder(leaf_size) * leaf_size + col.remainder(leaf_size)
    sentinel = side * side * leaf_size * leaf_size
    return torch.where(valid, cell * leaf_size * leaf_size + within, sentinel)


def _scatter_slots(tensor, slots, total):
    batch, heads, _, width = tensor.shape
    expanded = slots[:, None, :, None].expand(batch, heads, -1, width)
    return tensor.new_zeros(batch, heads, total + 1, width).scatter_add(2, expanded, tensor)[..., :total, :]


def _tree_readout(q_features, k_features, v, slots, geometry):
    """Uplift leaf sums; F_child=F_parent+interaction sums; contract by leaf.

    There is no subtraction and no B,H,N,neighbors,D or B,H,N,R,D gather.
    Sparse interaction matrices store <=27 entries per child at radius1.
    """
    batch, heads, count, features = q_features.shape
    side, cells, positions = geometry["side"], geometry["cell_count"], geometry["leaf_slots"]
    total = cells * positions
    dense_q = _scatter_slots(q_features, slots, total).reshape(batch, heads, cells, positions, features)
    dense_k = _scatter_slots(k_features, slots, total).reshape(batch, heads, cells, positions, features)
    dense_v = _scatter_slots(v, slots, total).reshape(batch, heads, cells, positions, v.shape[-1])
    # Explicit mass summary ensures padded values never contribute unit mass.
    summary = torch.cat((torch.matmul(dense_k.transpose(-2, -1), dense_v), dense_k.sum(-2).unsqueeze(-1)), -1)
    summaries = [summary]
    n = side
    while n > 1:
        summary = summary.reshape(batch, heads, n//2, 2, n//2, 2, features, -1).sum((3, 5))
        n //= 2
        summary = summary.reshape(batch, heads, n*n, features, -1)
        summaries.append(summary)
    summaries.reverse()
    far = torch.zeros_like(summaries[0])
    for level, own in zip(geometry["levels"], summaries[1:]):
        cell_count = own.shape[2]
        flattened = own.permute(2, 0, 1, 3, 4).reshape(cell_count, -1)
        interaction_sum = torch.sparse.mm(level["interaction"], flattened)
        interaction_sum = interaction_sum.reshape(cell_count, batch, heads, features, -1).permute(1, 2, 0, 3, 4)
        far = far.index_select(2, level["parent"]) + interaction_sum
    result = torch.matmul(dense_q, far).reshape(batch, heads, total, v.shape[-1] + 1)
    result = torch.cat((result, result.new_zeros(batch, heads, 1, result.shape[-1])), 2)
    return result.gather(2, slots[:, None, :, None].expand(batch, heads, count, result.shape[-1]))


def _local_scores(self, q, k, neighbors, neighbor_valid):
    chunks = []
    tile = int(getattr(self, "math_query_tile", 32))
    for start in range(0, q.shape[-2], tile):
        stop = min(start + tile, q.shape[-2])
        gather = self._gather_sources(k, neighbors[:, start:stop])
        scores = (q[:, :, start:stop, None, :] * gather).sum(-1) / math.sqrt(q.shape[-1])
        chunks.append(scores.masked_fill(~neighbor_valid[:, None, start:stop], -torch.inf))
    return torch.cat(chunks, -2)


def _local_tile(self, q, k, v, neighbors, neighbor_valid, query_shift, key_shift):
    nk = self._gather_sources(k, neighbors)
    nv = self._gather_sources(v, neighbors)
    scores = (q[:, :, :, None, :] * nk).sum(-1) / math.sqrt(q.shape[-1])
    pair = neighbor_valid[:, None]
    exponent = (scores - query_shift - key_shift).masked_fill(~pair, -torch.inf)
    weights = torch.exp(exponent)
    return torch.cat(((weights[..., None] * nv).sum(-2), weights.sum(-1, keepdim=True)), -1)


def _local_readout(self, q, k, v, neighbors, neighbor_valid, query_shift, key_shift):
    chunks = []
    tile = int(getattr(self, "math_query_tile", 32))
    checkpointed = bool(getattr(self, "math_checkpoint", True)) and torch.is_grad_enabled() and any(t.requires_grad for t in (q, k, v))
    for start in range(0, q.shape[-2], tile):
        stop = min(start + tile, q.shape[-2])
        args = (self, q[:, :, start:stop], k, v, neighbors[:, start:stop], neighbor_valid[:, start:stop], query_shift[:, :, start:stop], key_shift)
        # Checkpoint each tile independently so backward never holds all local
        # neighbor/value tensors simultaneously. No loop-captured tensor state.
        if checkpointed:
            out = checkpoint(_local_tile, *args, use_reentrant=False, preserve_rng_state=False)
        else:
            out = _local_tile(*args)
        chunks.append(out)
    return torch.cat(chunks, -2)


def tree_mix_heads(self, q, k, v, indices, valid, grid):
    batch, heads, count, dim = q.shape
    if count == 0:
        return q * 0 + k * 0 + v * 0
    dtype = torch.float64 if q.dtype == torch.float64 else torch.float32
    q, k, v = q.to(dtype), k.to(dtype), v.to(dtype)
    active = valid[:, None, :, None]
    q, k, v = [torch.where(active, t, torch.zeros_like(t)) for t in (q, k, v)]
    if not bool(torch.isfinite(q).all() & torch.isfinite(k).all() & torch.isfinite(v).all()):
        raise FloatingPointError("semantic-math-v1: nonfinite input at valid position")
    if getattr(self, "math_variant", "tree") == "centered_tree":
        k = center_keys(q, k, valid)
    _STATS["calls"] += 1; _STATS["valid_rows"] += int(valid.sum()) * heads
    active_rows = valid[:, None].expand(batch, heads, count)
    neighbors, neighbor_valid = self._packed_neighbors(indices, valid, grid)
    qs, ks = q * dim ** -0.25, k * dim ** -0.25
    normalizer = 0.5 * math.log(self.semantic_features)
    q_log = torch.einsum("bhnd,hrd->bhnr", qs, self.omega.to(dtype)) - qs.square().sum(-1, keepdim=True) / 2 - normalizer
    k_log = torch.einsum("bhnd,hrd->bhnr", ks, self.omega.to(dtype)) - ks.square().sum(-1, keepdim=True) / 2 - normalizer
    # Stabilizing shifts are constants in the differentiation, as in softmax.
    with torch.no_grad():
        local_scores = _local_scores(self, q, k, neighbors, neighbor_valid)
        key_shift = k_log.masked_fill(~active, -torch.inf).amax((-2, -1), keepdim=True)
        key_shift = torch.where(valid.any(-1)[:, None, None, None], key_shift, torch.zeros_like(key_shift))
        query_shift = torch.maximum(q_log.amax(-1, keepdim=True), local_scores.amax(-1, keepdim=True) - key_shift)
        query_shift = torch.where(active, query_shift, torch.zeros_like(query_shift))
    finite = torch.isfinite(q_log).all() & torch.isfinite(k_log).all() & torch.isfinite(local_scores.masked_fill(~neighbor_valid[:, None], 0)).all()
    if not bool(finite):
        suspect = active_rows
        output = q * 0 + k * 0 + v * 0
    else:
        q_features = torch.exp((q_log - query_shift).masked_fill(~active, -torch.inf))
        k_features = torch.exp((k_log - key_shift).masked_fill(~active, -torch.inf))
        geometry = _geometry(grid, self.local_leaf_size, self.local_radius, q.device, dtype)
        slots = _grid_slots(indices, valid, grid, self.local_leaf_size, geometry["side"])
        checkpointed = bool(getattr(self, "math_checkpoint", True)) and torch.is_grad_enabled() and any(t.requires_grad for t in (q, k, v))
        tree_fn = lambda a, b, c: _tree_readout(a, b, c, slots, geometry)
        local_fn = lambda a, b, c: _local_readout(self, a, b, c, neighbors, neighbor_valid, query_shift, key_shift)
        if checkpointed:
            far = checkpoint(tree_fn, q_features, k_features, v, use_reentrant=False, preserve_rng_state=False)
            near = local_fn(q, k, v)
        else:
            far = tree_fn(q_features, k_features, v)
            near = local_fn(q, k, v)
        combined = far + near
        numerator, denominator = combined[..., :-1], combined[..., -1]
        healthy = torch.isfinite(numerator).all(-1) & torch.isfinite(denominator) & (denominator >= math.sqrt(torch.finfo(dtype).tiny))
        suspect = active_rows & ~healthy
        regular = active_rows & ~suspect
        numerator = torch.where(regular[..., None], numerator, torch.zeros_like(numerator))
        denominator = torch.where(regular, denominator, torch.ones_like(denominator))
        output = numerator / denominator[..., None]
    if bool(suspect.any()):
        coordinates = suspect.nonzero(as_tuple=False)
        repaired = _stable_module()._log_space_rows(self, q, k, v, indices, valid, grid, coordinates)
        output = output.index_put(tuple(coordinates.unbind(-1)), repaired)
        _STATS["fallback_calls"] += 1; _STATS["fallback_rows"] += len(coordinates)
    if not bool(torch.isfinite(output).all()):
        raise FloatingPointError("semantic-math-v1: nonfinite result after direct fallback")
    return torch.where(active, output, torch.zeros_like(output))


def configure_model(model, variant, feature_counts=None):
    """Bind per-instance methods; no global monkey patch or state-dict ambiguity.

    feature_counts: sequence indexed by layer, mapping layer-index to rank, or
    None. Context and EMA layers with equal feature_seed receive identical banks.
    Call before optimizer construction/checkpoint loading for a new rank layout.
    """
    if variant not in _VARIANTS:
        raise ValueError(f"Unknown variant {variant!r}; expected {_VARIANTS}")
    mixers = [(name, module) for name, module in model.named_modules() if isinstance(module, SemanticLocalMixer)]
    layer_seeds = sorted(set(module.feature_seed for _, module in mixers))
    descriptions = []
    for name, module in mixers:
        layer = layer_seeds.index(module.feature_seed)
        if feature_counts is not None:
            rank = int(feature_counts.get(layer, module.semantic_features) if isinstance(feature_counts, dict) else feature_counts[layer])
            if rank < 1:
                raise ValueError("Feature counts must be positive")
            if rank != module.semantic_features:
                module.omega = _orthogonal_gaussian_features(module.heads, rank, module.dim // module.heads, module.feature_seed).to(device=module.omega.device, dtype=module.omega.dtype)
                module.semantic_features = rank
        module.math_variant = variant
        module.math_query_tile = 32
        module.math_checkpoint = True
        function = _stable_module().stable_mix_heads if variant == "stable" else centered_mix_heads if variant == "centered" else tree_mix_heads
        module.mix_heads = types.MethodType(function, module)
        module.backend = backend + ":" + variant
        descriptions.append({"name": name, "layer": layer, "features": module.semantic_features, "feature_seed": module.feature_seed})
    return {"backend": backend, "variant": variant, "mixers": descriptions}
